# Master Qwen Judge — All 540 judgements

Reads configs saved by `master_run_pipeline.ipynb` and sends each to Qwen for judgement in three modes (strict / lenient / configurable).

**Resume-safe.** Stop the kernel any time — when you re-run cell 4 it skips judgements already in the CSV.

**Cost:** €0 (Qwen on university server). **Time:** ~30-60 minutes.

## Cell 1 — Setup

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup. Reads configs from notebook 1, sends to Qwen.
# ═══════════════════════════════════════════════════════════════════
import sys, os, time, json, csv, traceback
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CANDIDATES = [
    "/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch",
    os.path.abspath(".."), os.getcwd(),
]
PROJECT_ROOT = next(
    (p for p in CANDIDATES if os.path.isfile(os.path.join(p, "generate_visualization.py"))), None
)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from retrieve_data import retrieve_data
MD_TABLE = retrieve_data(None, type="test")
QUESTION = ("Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? "
            "Provide a detailed analysis and a comprehensive visualization.")

ROOT_OUT = "master_run_results"
MASTER_CSV    = f"{ROOT_OUT}/master_pipeline_runs.csv"
JUDGE_CSV     = f"{ROOT_OUT}/master_qwen_judge.csv"
JUDGE_DIR     = f"{ROOT_OUT}/qwen_verdicts"
os.makedirs(JUDGE_DIR, exist_ok=True)

# Qwen judge endpoint
from openai import OpenAI
qwen = OpenAI(
    base_url="http://hal9000.skim.th-owl.de:11877/v1",
    api_key="dummy",
    timeout=120,
)
QWEN_MODEL = "Qwen3-30B-A3B-Instruct"

# verify pipeline output exists
assert os.path.exists(MASTER_CSV), f"{MASTER_CSV} not found — run master_run_pipeline.ipynb first"
df_pipeline = pd.read_csv(MASTER_CSV)
print(f"Loaded {len(df_pipeline)} pipeline runs from {MASTER_CSV}")
print(f"Will judge each with Qwen in 3 modes = {len(df_pipeline)*3} judge calls total")


Loaded 180 pipeline runs from master_run_results/master_pipeline_runs.csv
Will judge each with Qwen in 3 modes = 540 judge calls total


## Cell 2 — Three judge prompts

In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Three judge prompts (strict / lenient / configurable)
# ═══════════════════════════════════════════════════════════════════

JUDGE_STRICT = """You are a STRICT chart configuration validator.

You will receive: (1) the user query, (2) the data table, (3) a JSON visualization config produced by an LLM.

Evaluate against these 8 rules. ALL must pass for verdict=PASS:
1. CHART_TYPE_MATCH: charttype matches user intent (monthly→bar/line; annual→bar/column).
2. TITLE_QUALITY: title mentions Teckentrup, JVA segment, and time range; not generic.
3. DATA_COMPLETENESS: data covers the requested time range.
4. DATA_GROUPING: data series are clearly labelled (e.g. "2021","2022","2023","2024").
5. AXIS_LABELS: xlabel and ylabel are meaningful.
6. ANNOTATIONS_PRESENT: annotations exist and are specific (not just "data").
7. SCALE_REASONABLE: data range looks proportional, no obvious clipping.
8. NO_HALLUCINATION: data values exist in the source table.

Respond with VALID JSON ONLY (no markdown, no preamble):
{"verdict":"PASS|FAIL","checks":{"chart_type_match":true|false,"title_quality":true|false,"data_completeness":true|false,"data_grouping":true|false,"axis_labels":true|false,"annotations_present":true|false,"scale_reasonable":true|false,"no_hallucination":true|false},"score":0,"issues":[],"fix_suggestion":""}

verdict=PASS only if all 8 checks pass. score = sum of true checks (0-8)."""

JUDGE_LENIENT = """You are a LENIENT chart configuration validator. Flag only MAJOR issues.

Major issues only:
A. WRONG_DATA: data doesn't address the question (wrong subject/segment/metric).
B. EMPTY_OUTPUT: data array empty or all zero.
C. SEVERELY_WRONG_CHART: e.g. pie for time series, scatter for categorical.
D. HALLUCINATION: values not in source table.

Respond with VALID JSON ONLY:
{"verdict":"PASS|FAIL","checks":{"wrong_data":true|false,"empty_output":true|false,"severely_wrong_chart":true|false,"hallucination":true|false},"score":0,"issues":[],"fix_suggestion":""}

verdict=FAIL only if ANY major issue (A-D) present. score = 4 minus issues."""

JUDGE_CONFIG = """You are a chart configuration validator with threshold T.

Score these 8 dimensions, each 0.0 to 1.0:
1. chart_type_match
2. title_specificity
3. data_completeness
4. data_grouping_quality
5. axis_label_quality
6. annotation_quality (1.0 if not needed for query type)
7. scale_reasonableness
8. data_fidelity

Compute mean_score. verdict=PASS if mean_score>=T, else FAIL.

Respond with VALID JSON ONLY:
{"threshold":0.7,"dimensions":{"chart_type_match":0.0,"title_specificity":0.0,"data_completeness":0.0,"data_grouping_quality":0.0,"axis_label_quality":0.0,"annotation_quality":0.0,"scale_reasonableness":0.0,"data_fidelity":0.0},"mean_score":0.0,"verdict":"PASS|FAIL","issues":[],"fix_suggestion":""}"""

JUDGE_PROMPTS = {"strict": JUDGE_STRICT, "lenient": JUDGE_LENIENT, "configurable": JUDGE_CONFIG}

print("Three judge prompts ready.")


Three judge prompts ready.


## Cell 3 — Judge function

In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Judge function (resume-safe)
# ═══════════════════════════════════════════════════════════════════

def call_qwen_judge(question, md_table, gpt_config, mode, threshold=0.7):
    """One Qwen judge call. Returns (verdict_dict, latency_s, error_str)."""
    sys_p = JUDGE_PROMPTS[mode]
    user_msg = f"""USER QUERY:
{question}

DATA TABLE (excerpt):
{md_table[:1500]}{'...' if len(md_table) > 1500 else ''}

JSON CONFIG TO EVALUATE:
{json.dumps(gpt_config, indent=2, default=str)}
"""
    if mode == "configurable":
        user_msg += f"\n\nTHRESHOLD T = {threshold}"

    t0 = time.perf_counter()
    try:
        resp = qwen.chat.completions.create(
            model=QWEN_MODEL,
            messages=[{"role":"system","content":sys_p}, {"role":"user","content":user_msg}],
            max_tokens=800, temperature=0.0,
        )
        lat = round(time.perf_counter()-t0, 3)
        raw = resp.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"): raw = raw[4:]
            raw = raw.strip()
        if "{" in raw:
            raw = raw[raw.index("{"):]
        if raw.endswith("```"):
            raw = raw[:-3].strip()
        return json.loads(raw), lat, None
    except json.JSONDecodeError as e:
        return None, round(time.perf_counter()-t0,3), f"JSON: {str(e)[:80]} | raw: {raw[:120]}"
    except Exception as e:
        return None, round(time.perf_counter()-t0,3), f"{type(e).__name__}: {str(e)[:120]}"

print("Qwen judge function ready.")


Qwen judge function ready.


## Cell 4 — Master judge loop (resume-safe)

In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Master judge loop (resume-safe)
# 540 calls total (180 runs × 3 modes), all free.
# ═══════════════════════════════════════════════════════════════════

JUDGE_FIELDS = [
    "scenario","run","mode","timestamp","judge_latency_s",
    "verdict","score","mean_score","n_issues","issues","fix_suggestion","error",
]

# load already-completed
done = set()
if os.path.exists(JUDGE_CSV):
    df_done = pd.read_csv(JUDGE_CSV)
    for _, r in df_done.iterrows():
        done.add((r["scenario"], int(r["run"]), r["mode"]))
    print(f"Found {len(done)} judgements already completed. Will skip.")
else:
    with open(JUDGE_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=JUDGE_FIELDS).writeheader()
    print("Starting fresh judge CSV")

attempted = 0
total_planned = len(df_pipeline) * 3

for _, prow in df_pipeline.iterrows():
    sid = prow["scenario"]; run_idx = int(prow["run"])
    cfg_path = prow["config_path"]
    if not isinstance(cfg_path, str) or not os.path.exists(cfg_path):
        print(f"  {sid} run{run_idx:02d}: config missing — skip")
        continue

    # load saved config
    with open(cfg_path) as f:
        cfg = json.load(f)
    if "_error" in cfg:
        # pipeline failed — judge won't help, mark as N/A
        for mode in ["strict","lenient","configurable"]:
            if (sid, run_idx, mode) in done: continue
            with open(JUDGE_CSV, "a", newline="") as f:
                csv.DictWriter(f, fieldnames=JUDGE_FIELDS).writerow({
                    "scenario":sid,"run":run_idx,"mode":mode,
                    "timestamp":time.strftime("%Y-%m-%dT%H:%M:%S"),
                    "judge_latency_s":0,"verdict":"N/A","score":"","mean_score":"",
                    "n_issues":0,"issues":"","fix_suggestion":"",
                    "error":"pipeline_failed",
                })
        continue

    print(f"\n  {sid} run{run_idx:02d}:")
    for mode in ["strict","lenient","configurable"]:
        if (sid, run_idx, mode) in done:
            print(f"    {mode:<13} [skip]")
            continue
        attempted += 1
        print(f"    {mode:<13} ...", end="", flush=True)
        verdict, lat, err = call_qwen_judge(QUESTION, MD_TABLE, cfg, mode)

        if verdict:
            v = verdict.get("verdict","?")
            score = verdict.get("score","")
            mean_score = verdict.get("mean_score","")
            issues = verdict.get("issues",[]) or []
            fix = verdict.get("fix_suggestion","") or ""
            row = {
                "scenario":sid, "run":run_idx, "mode":mode,
                "timestamp":time.strftime("%Y-%m-%dT%H:%M:%S"),
                "judge_latency_s":lat, "verdict":v,
                "score":score, "mean_score":mean_score,
                "n_issues":len(issues),
                "issues":" | ".join(str(i) for i in issues)[:500],
                "fix_suggestion":fix[:300], "error":"",
            }
            # save full verdict json
            with open(f"{JUDGE_DIR}/{sid}_run{run_idx:02d}_{mode}.json","w") as f:
                json.dump(verdict, f, indent=2)
            print(f" {v} ({lat:.1f}s, {len(issues)} issues)")
        else:
            row = {"scenario":sid,"run":run_idx,"mode":mode,
                   "timestamp":time.strftime("%Y-%m-%dT%H:%M:%S"),
                   "judge_latency_s":lat,"verdict":"ERROR","score":"","mean_score":"",
                   "n_issues":0,"issues":"","fix_suggestion":"","error":err[:200] if err else ""}
            print(f" ERROR ({lat:.1f}s) {err[:60] if err else ''}")

        with open(JUDGE_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=JUDGE_FIELDS).writerow(row)

print(f"\n{'='*60}")
print(f"  Judging done. {attempted} judge calls this session.")
print(f"  Master CSV: {JUDGE_CSV}")
print(f"  Verdicts:   {JUDGE_DIR}/")


Starting fresh judge CSV

  S3 run01:
    strict        ... ERROR (0.6s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    lenient       ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    configurable  ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 

  S3 run02:
    strict        ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    lenient       ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    configurable  ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 

  S3 run03:
    strict        ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    lenient       ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 
    configurable  ... ERROR (0.1s) NotFoundError: Error code: 404 - {'error': {'message': 'The 

  S3 run04:
    strict        ... ERROR (0.1s) NotFoundError: Error cod

## Cell 5 — Status summary

In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Quick judge summary
# ═══════════════════════════════════════════════════════════════════

dfj = pd.read_csv(JUDGE_CSV)
dfj["judge_latency_s"] = pd.to_numeric(dfj["judge_latency_s"], errors="coerce")

print(f"\n{'='*70}\n  QWEN JUDGE COMPLETION STATUS\n{'='*70}")
print(f"  Total judgements:  {len(dfj)}")
print(f"  Across modes:")
for mode in ["strict","lenient","configurable"]:
    sub = dfj[dfj["mode"]==mode]
    n_pass = (sub["verdict"]=="PASS").sum()
    n_fail = (sub["verdict"]=="FAIL").sum()
    n_err  = (sub["verdict"]=="ERROR").sum()
    n_na   = (sub["verdict"]=="N/A").sum()
    lat_mean = sub["judge_latency_s"].mean()
    print(f"    {mode:<13}: {len(sub):>4}  PASS:{n_pass:>3}  FAIL:{n_fail:>3}  "
          f"ERROR:{n_err:>3}  N/A:{n_na:>3}  mean_lat:{lat_mean:>5.1f}s")

# per-scenario quick view
print(f"\n  PER-SCENARIO PASS COUNTS (PASS / total per mode)")
print(f"  {'Scenario':<6} {'Strict':>10} {'Lenient':>10} {'Config':>10}")
print("-"*45)
for sid in sorted(dfj["scenario"].unique()):
    parts = []
    for mode in ["strict","lenient","configurable"]:
        sub = dfj[(dfj["scenario"]==sid) & (dfj["mode"]==mode)]
        n_pass = (sub["verdict"]=="PASS").sum()
        n_total = len(sub)
        parts.append(f"{n_pass}/{n_total}")
    print(f"  {sid:<6} {parts[0]:>10} {parts[1]:>10} {parts[2]:>10}")
print("="*70)



  QWEN JUDGE COMPLETION STATUS
  Total judgements:  540
  Across modes:
    strict       :  180  PASS:  0  FAIL:  0  ERROR:180  N/A:  0  mean_lat:  0.1s
    lenient      :  180  PASS:  0  FAIL:  0  ERROR:180  N/A:  0  mean_lat:  0.1s
    configurable :  180  PASS:  0  FAIL:  0  ERROR:180  N/A:  0  mean_lat:  0.1s

  PER-SCENARIO PASS COUNTS (PASS / total per mode)
  Scenario     Strict    Lenient     Config
---------------------------------------------
  S0           0/10       0/10       0/10
  S1           0/10       0/10       0/10
  S2           0/10       0/10       0/10
  S3           0/10       0/10       0/10
  S4           0/10       0/10       0/10
  S4b          0/10       0/10       0/10
  S5           0/10       0/10       0/10
  S5b          0/10       0/10       0/10
  S6           0/10       0/10       0/10
  S7           0/10       0/10       0/10
  S8           0/10       0/10       0/10
  S9           0/10       0/10       0/10
  SA1          0/10       0/10       0